# De l'ACP aux auto-encodeurs profonds avec PyTorch Lightning

Un auto-encodeur comprime son entrée en un code de petite taille (l'encodeur), puis la reconstruit depuis ce code (le décodeur) : sa cible est son entrée elle-même. Ce notebook en entraîne plusieurs sur MNIST (60 000 images d'entraînement et 10 000 de test, des chiffres manuscrits 28 × 28 en niveaux de gris), du plus simple au plus profond :

1. Un auto-encodeur linéaire
2. L'auto-encodeur linéaire face à l'ACP
3. Un auto-encodeur profond
4. L'espace latent
5. La détection d'anomalies
6. La taille du code
7. Pour aller plus loin : un auto-encodeur épars

Chaque partie réutilise les modèles des précédentes ; la dernière est facultative. Pas besoin de GPU : le notebook tourne en 20 à 30 minutes sur un processeur, dont la moitié pour la partie 6, qui entraîne six auto-encodeurs.

## Installation et imports

In [ ]:
!pip install -q lightning

In [ ]:
from collections import OrderedDict

import lightning
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from sklearn.decomposition import PCA
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

lightning.seed_everything(0)

## Chargement de MNIST

In [ ]:
train_data = torchvision.datasets.MNIST("data", train=True, download=True)
test_data = torchvision.datasets.MNIST("data", train=False, download=True)
input_dim = 28 * 28
# Pixels dans [0, 1], comme la sortie d'une sigmoïde : ni centrés, ni réduits
X_train = train_data.data.reshape(-1, input_dim).float() / 255
y_train = train_data.targets
X_test = test_data.data.reshape(-1, input_dim).float() / 255
y_test = test_data.targets
print(X_train.shape, X_test.shape)


def show_images(*rows: torch.Tensor, titles: list | None = None) -> None:
  # Affiche des rangées d'images 28 × 28, une rangée par tableau
  n = len(rows[0])
  _, axes = plt.subplots(
    len(rows), n, figsize=(n * 1.2, len(rows) * 1.3), squeeze=False
  )
  for row, images in zip(axes, rows):
    for ax, image in zip(row, images):
      ax.imshow(image.reshape(28, 28), cmap="gray_r")
      ax.axis("off")
  for ax, title in zip(axes[0], [] if titles is None else titles):
    ax.set_title(title)
  plt.show()


show_images(X_train[:10], titles=y_train[:10].tolist())

## Les outils de l'entraînement

Ils sont fournis, et servent à tout le notebook :

- `AutoEncoder`, un `LightningModule` qui enveloppe un réseau et une perte : la cible de chaque image est l'image elle-même ; il initialise les poids du réseau comme Keras, ce qui évite que des unités ReLU du code meurent ;
- `train(model, max_epochs, name)`, qui entraîne un `AutoEncoder` par lots de 256 images, mesure la perte de validation sur les images de test et rend sa dernière valeur ;
- `predict(network, X)`, qui applique un réseau à toutes les lignes de `X`, lot par lot, sans calculer de gradient.

In [ ]:
class AutoEncoder(lightning.LightningModule):
  """Lightning wrapper: the input is also the target to reconstruct."""

  def __init__(
    self, model: nn.Module, loss: nn.Module, learning_rate: float = 1e-3
  ) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model", "loss"])
    self.model = model
    self.loss = loss
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, input_dim)
    # Initialisation de Glorot, celle de Keras : avec celle de PyTorch, la plupart
    # des unités ReLU d'un petit code meurent (nulles pour toutes les images)
    for layer in model.modules():
      if isinstance(layer, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_uniform_(layer.weight)
        nn.init.zeros_(layer.bias)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor], stage: str) -> torch.Tensor:
    (images,) = batch
    loss = self.loss(self(images), images)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def training_step(self, batch: tuple[torch.Tensor], batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(
    self, batch: tuple[torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)


train_loader = DataLoader(TensorDataset(X_train), batch_size=256, shuffle=True)
# La perte de validation se calcule sur les images de test
val_loader = DataLoader(TensorDataset(X_test), batch_size=256)


def train(
  model: AutoEncoder, max_epochs: int, name: str, verbose: bool = True
) -> float:
  """Train model on MNIST and return its last validation loss."""
  trainer = lightning.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("logs", name=name),
    # La barre de progression de tqdm, plutôt que celle de rich que Lightning
    # choisit quand rich est installé (Colab) : elle a figé des entraînements
    # (Lightning refuse ce callback quand enable_progress_bar est False)
    callbacks=[TQDMProgressBar()] if verbose else [],
    enable_checkpointing=False,
    enable_progress_bar=verbose,
    enable_model_summary=False,
  )
  trainer.fit(model, train_loader, val_loader)
  return trainer.callback_metrics["val_loss"].item()


@torch.no_grad()
def predict(model: nn.Module, X: torch.Tensor, batch_size: int = 256) -> torch.Tensor:
  """Apply a network to every row of X, batch by batch."""
  device = next(model.parameters()).device
  model.eval()
  return torch.cat([model(batch.to(device)).cpu() for batch in X.split(batch_size)])

## 1. Un auto-encodeur linéaire

Sans fonction d'activation, l'encodeur et le décodeur ne sont que des applications linéaires (plus un biais). Construisez :

- `encoder` : un `nn.Sequential` d'une seule couche `nn.Linear`, de `input_dim` entrées vers `encoding_dim = 20` sorties ;
- `decoder` : un `nn.Sequential` d'une seule couche `nn.Linear`, de `encoding_dim` entrées vers `input_dim` sorties ;
- `autoencoder`, un `nn.Sequential` qui enchaîne les deux, sous les noms `encoder` et `decoder` ;
- `model`, un `AutoEncoder` qui enveloppe `autoencoder` avec la perte `nn.MSELoss()`, dont vous affichez le résumé avec `ModelSummary`.

La cellule après la solution l'entraîne.

**Documentation utile**

- [`torch.nn.Sequential`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
- [`torch.nn.Linear`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Linear(in_features, out_features)`, sans activation après : une couche linéaire
- `nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))` : un `nn.Sequential` construit sur un `OrderedDict` nomme ses sous-modules, qu'on retrouve ensuite en `autoencoder.encoder` et `autoencoder.decoder` ; `encoder` et `decoder` restent utilisables seuls
- `print(ModelSummary(model, max_depth=2))` doit afficher 15,7 K paramètres pour l'encodeur (784 × 20 poids + 20 biais) et 16,5 K pour le décodeur (20 × 784 + 784)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
encoding_dim = 20
encoder = nn.Sequential(nn.Linear(..., ...))
decoder = ...
autoencoder = nn.Sequential(OrderedDict(encoder=..., decoder=...))
model = AutoEncoder(autoencoder, loss=...)
print(ModelSummary(model, max_depth=2))
```

### Solution

In [ ]:
encoding_dim = 20
encoder = nn.Sequential(nn.Linear(input_dim, encoding_dim))
decoder = nn.Sequential(nn.Linear(encoding_dim, input_dim))
autoencoder = nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))
model = AutoEncoder(autoencoder, loss=nn.MSELoss())
print(ModelSummary(model, max_depth=2))

### Entraînement

La cible de chaque image est l'image elle-même, et la perte de validation se calcule sur les images de test.

In [ ]:
val_loss = train(model, max_epochs=10, name="linear")
print(f"Perte de validation : {val_loss:.4f}")
linear_decoder = decoder  # gardé pour la partie 2
X_test_linear = predict(autoencoder, X_test)
show_images(X_test[:10], X_test_linear[:10], titles=y_test[:10].tolist())

## 2. L'auto-encodeur linéaire face à l'ACP

La slide « Auto-encodeur linéaire et ACP » l'affirme : entraîné avec la MSE, un auto-encodeur linéaire à code de taille k retrouve le sous-espace des k premières composantes de l'ACP, mais pas ses axes. Vérifiez-le :

- ajustez une `PCA` à `encoding_dim` composantes sur `X_train`, reconstruisez `X_test` avec elle, et comparez l'erreur quadratique moyenne de sa reconstruction à celle de l'auto-encodeur (`X_test_linear`, calculé par la cellule précédente) ;
- les 20 directions du décodeur sont les colonnes du poids `weight` (784 × 20) de sa couche `nn.Linear` : mesurez, pour chacune, la part de sa norme qui tombe dans le sous-espace de l'ACP ;
- mesurez enfin le plus grand cosinus, en valeur absolue, entre deux directions du décodeur, puis entre deux composantes de l'ACP.

Les deux erreurs sont-elles proches ? Les directions du décodeur sont-elles dans le sous-espace de l'ACP ? Sont-elles orthogonales, comme les composantes ?

**Documentation utile**

- [`sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [`torch.Tensor.numpy`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.numpy.html)
- [`numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- scikit-learn travaille sur des tableaux NumPy : `X_train.numpy()` en donne un, qui partage la mémoire du tenseur
- `pca = PCA(encoding_dim).fit(X_train.numpy())`, puis `pca.inverse_transform(pca.transform(X_test.numpy()))` : la reconstruction par l'ACP ; l'erreur : `np.mean((X_test.numpy() - X_test_pca) ** 2)`
- `nn.Linear` range ses poids en (sorties, entrées) : `linear_decoder[0].weight` est de forme (784, 20), une direction par colonne ; `W = linear_decoder[0].weight.detach().numpy().T` les met en lignes, de forme (20, 784), comme `pca.components_` (`.detach()` sépare le tenseur du calcul des gradients, ce que `.numpy()` exige)
- les lignes de `pca.components_` sont orthonormées : `W @ pca.components_.T` donne les coordonnées de chaque direction dans la base de l'ACP ; la norme de chaque ligne de ce produit (`np.linalg.norm(..., axis=1)`), divisée par celle de la ligne de `W`, est la part cherchée (1 : la direction est entièrement dans le sous-espace)
- les cosinus : normalisez les lignes, `V / np.linalg.norm(V, axis=1, keepdims=True)`, puis `V @ V.T` ; hors de la diagonale (`~np.eye(20, dtype=bool)`), 0 pour deux directions orthogonales

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
pca = PCA(...).fit(...)
X_test_pca = ...
pca_error = np.mean((X_test.numpy() - X_test_pca) ** 2)
linear_error = ...
print(f"Erreur : ACP {pca_error:.4f}, AE linéaire {linear_error:.4f}")

W = linear_decoder[0].weight.detach().numpy().T
share = ...
print(f"Part dans le sous-espace de l'ACP : {share.min():.3f} à {share.max():.3f}")


def max_cosine(V: np.ndarray) -> float:
  V = ...
  cosines = ...
  return np.abs(cosines[~np.eye(len(V), dtype=bool)]).max()


print(f"Plus grand cosinus : décodeur {max_cosine(W):.2f}, ACP {max_cosine(...):.2f}")
```

### Solution

In [ ]:
pca = PCA(encoding_dim).fit(X_train.numpy())
X_test_pca = pca.inverse_transform(pca.transform(X_test.numpy()))
pca_error = np.mean((X_test.numpy() - X_test_pca) ** 2)
linear_error = ((X_test - X_test_linear) ** 2).mean().item()
print(f"Erreur : ACP {pca_error:.4f}, AE linéaire {linear_error:.4f}")

W = linear_decoder[0].weight.detach().numpy().T  # (20, 784)
share = np.linalg.norm(W @ pca.components_.T, axis=1) / np.linalg.norm(W, axis=1)
print(f"Part dans le sous-espace de l'ACP : de {share.min():.3f} à {share.max():.3f}")


def max_cosine(V: np.ndarray) -> float:
  V = V / np.linalg.norm(V, axis=1, keepdims=True)
  cosines = V @ V.T
  return np.abs(cosines[~np.eye(len(V), dtype=bool)]).max()


print(
  f"Plus grand cosinus : décodeur {max_cosine(W):.2f}, ACP {max_cosine(pca.components_):.2f}"
)

Les deux erreurs sont presque égales (environ 0,024) : l'auto-encodeur linéaire fait aussi bien que l'ACP, mais pas mieux, puisque l'ACP donne la meilleure reconstruction linéaire possible à 20 dimensions. Ses 20 directions sont presque entièrement dans le sous-espace de l'ACP (de 93 % à presque 100 % de leur norme) : il a trouvé le même sous-espace, aux imprécisions de l'entraînement près. Mais elles ne sont pas orthogonales (le plus grand cosinus vaut environ 0,25, contre 0 pour l'ACP), ni triées par variance expliquée : rien dans la perte ne l'exige. L'ACP donne cette base en plus, et sans entraînement.

## 3. Un auto-encodeur profond

Ajoutez une couche cachée non linéaire de part et d'autre du code :

- `encoder` : une `nn.Linear` de `input_dim` vers 128, puis une de 128 vers `encoding_dim`, chacune suivie d'une ReLU ;
- `decoder` : une `nn.Linear` de `encoding_dim` vers 128 suivie d'une ReLU, puis la couche de sortie, de 128 vers `input_dim`, suivie d'une activation ;
- `autoencoder`, qui les enchaîne sous les noms `encoder` et `decoder`, et `model`, l'`AutoEncoder` qui l'enveloppe.

Choisissez l'activation de la couche de sortie et la perte, sachant que les pixels sont dans [0, 1], et justifiez votre choix en une ligne de commentaire. La cellule après la solution l'entraîne, puis compare son erreur à celles de l'ACP et de l'auto-encodeur linéaire.

**Documentation utile**

- [`torch.nn.Sigmoid`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sigmoid.html)
- [`torch.nn.BCELoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCELoss.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Linear(input_dim, 128), nn.ReLU()` : une couche cachée
- la sortie : `nn.Sigmoid()`, qui rend des valeurs dans ]0, 1[, comme les pixels ; une sortie linéaire peut rendre des pixels négatifs ou plus grands que 1
- la perte : `nn.BCELoss()`, qui traite chaque pixel comme une probabilité (la cible peut être entre 0 et 1, pas seulement 0 ou 1) et attend des sorties dans ]0, 1[, ou `nn.MSELoss()`, qui fonctionne aussi
- `print(ModelSummary(model, max_depth=2))` : 103 K paramètres pour l'encodeur (103 060), 103 K pour le décodeur (103 824)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
encoder = nn.Sequential(
  nn.Linear(input_dim, 128), nn.ReLU(), nn.Linear(..., ...), nn.ReLU()
)
decoder = nn.Sequential(..., ..., nn.Linear(128, input_dim), ...)
autoencoder = nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))
model = AutoEncoder(autoencoder, loss=...)  # parce que ...
print(ModelSummary(model, max_depth=2))
```

### Solution

In [ ]:
encoder = nn.Sequential(
  nn.Linear(input_dim, 128),
  nn.ReLU(),
  nn.Linear(128, encoding_dim),
  nn.ReLU(),
)
decoder = nn.Sequential(
  nn.Linear(encoding_dim, 128),
  nn.ReLU(),
  nn.Linear(128, input_dim),
  nn.Sigmoid(),  # sortie dans ]0, 1[, comme les pixels
)
autoencoder = nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))
# Chaque pixel cible, dans [0, 1], est traité comme une probabilité
model = AutoEncoder(autoencoder, loss=nn.BCELoss())
print(ModelSummary(model, max_depth=2))

### Entraînement et comparaison

In [ ]:
train(model, max_epochs=20, name="deep")
X_test_pred = predict(autoencoder, X_test)
deep_error = ((X_test - X_test_pred) ** 2).mean().item()
print(
  f"Erreur quadratique moyenne : ACP {pca_error:.4f}, "
  f"AE linéaire {linear_error:.4f}, AE profond {deep_error:.4f}"
)
# En haut les originaux, puis l'AE linéaire, puis l'AE profond
show_images(
  X_test[:10], X_test_linear[:10], X_test_pred[:10], titles=y_test[:10].tolist()
)

Avec le même code de 20 nombres, l'auto-encodeur profond reconstruit nettement mieux que l'ACP et que l'auto-encodeur linéaire (une erreur d'environ 0,012, deux fois moindre) : ses couches non linéaires suivent des surfaces courbes, hors de portée d'un sous-espace.

## 4. L'espace latent

L'encodeur seul transforme chaque image de test en son code, un point d'un espace à `encoding_dim` dimensions : l'espace latent.

In [ ]:
codes = predict(encoder, X_test)
print(codes.shape)

### Où vivent les codes ?

Le décodeur seul pourrait-il fabriquer de nouveaux chiffres à partir de codes tirés au hasard ? Regardez d'abord où vivent les vrais codes, puis essayez :

- pour chacune des 20 dimensions, affichez le minimum, la moyenne et le maximum des `codes`, et comptez les dimensions qui valent 0 pour toutes les images ;
- décodez 10 codes de bruit blanc, tirés uniformément dans [0, 1] sur chaque dimension, puis 10 codes tirés uniformément entre le minimum et le maximum de chaque dimension, et affichez les deux rangées d'images avec `show_images`.

Les codes ressemblent-ils à du bruit uniforme dans [0, 1] ? Les images décodées ressemblent-elles à des chiffres ? Pourquoi ?

**Documentation utile**

- [`torch.min`](https://docs.pytorch.org/docs/stable/generated/torch.min.html)
- [`torch.rand`](https://docs.pytorch.org/docs/stable/generated/torch.rand.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `codes.min(dim=0).values`, `codes.mean(dim=0)`, `codes.max(dim=0).values` : une valeur par dimension (`min` et `max` avec `dim` rendent aussi les indices, dans `.indices`) ; `.numpy().round(1)` pour les afficher
- une dimension qui vaut 0 pour toutes les images : `high == 0` (la ReLU du code ne rend jamais moins de 0), puis `.sum()`
- `generator = torch.Generator().manual_seed(0)`, puis `torch.rand(10, encoding_dim, generator=generator)` : des valeurs uniformes dans [0, 1[ ; `low + (high - low) * torch.rand(...)` les ramène entre `low` et `high`, dimension par dimension
- `predict(decoder, noise)` rend 10 images de 784 pixels, que `show_images` affiche

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
low, high = ..., ...
print("Min :", low.numpy().round(1))
print("Moyenne :", ...)
print("Max :", ...)
print("Dimensions mortes :", int(...))

generator = torch.Generator().manual_seed(0)
white_noise = torch.rand(...)
in_range = low + (high - low) * ...
show_images(predict(decoder, white_noise), predict(decoder, ...))
```

#### Solution

In [ ]:
low, high = codes.min(dim=0).values, codes.max(dim=0).values
print("Min :", low.numpy().round(1))
print("Moyenne :", codes.mean(dim=0).numpy().round(1))
print("Max :", high.numpy().round(1))
print("Dimensions mortes :", int((high == 0).sum()))

generator = torch.Generator().manual_seed(0)
white_noise = torch.rand(10, encoding_dim, generator=generator)
in_range = low + (high - low) * torch.rand(10, encoding_dim, generator=generator)
# En haut le bruit blanc, en bas les codes tirés dans leur plage
show_images(predict(decoder, white_noise), predict(decoder, in_range))

Les codes ne ressemblent pas à du bruit uniforme dans [0, 1] : la ReLU les rend positifs ou nuls, leur plage change d'une dimension à l'autre et va jusqu'à plusieurs dizaines. Aucune dimension n'est morte ici : nulle pour toutes les images, elle ne porterait aucune information. Le bruit blanc tombe donc dans un coin de l'espace latent, près de l'origine, que le décodeur n'a presque jamais vu : il rend des taches qui se ressemblent toutes. Tirés dans leur plage, les codes ne donnent pas mieux, des morceaux de traits qui ne forment pas de chiffre : les vrais codes n'occupent qu'une petite partie de ce pavé, car leurs dimensions sont liées entre elles. Un auto-encodeur compresse, mais ne génère pas : son décodeur ne sait décoder que les codes que produit l'encodeur. Pour générer, il faut imposer aux codes une loi connue dans laquelle on peut tirer : c'est ce que fait l'auto-encodeur variationnel (slide « Architecture d'un VAE »).

### Les centroïdes des chiffres

Les codes ne sont pas dispersés au hasard : ceux d'un même chiffre se regroupent. Le code moyen de chaque chiffre, son centroïde, décodé, en donne une image type.

In [ ]:
centroids = torch.stack([codes[y_test == digit].mean(dim=0) for digit in range(10)])
show_images(predict(decoder, centroids), titles=list(range(10)))

### Interpoler entre deux centroïdes

Les codes régulièrement espacés sur le segment qui relie deux centroïdes, une fois décodés, montrent comment le décodeur passe d'un chiffre à l'autre. Essayez d'autres paires.

In [ ]:
def latent_walk(start: int, end: int, n: int = 12) -> None:
  # n codes régulièrement espacés, du centroïde de start à celui de end
  steps = torch.linspace(0, 1, n)[:, None]
  walk = centroids[start] + steps * (centroids[end] - centroids[start])
  show_images(predict(decoder, walk))


latent_walk(7, 1)
latent_walk(3, 8)

## 5. La détection d'anomalies

Un auto-encodeur reconstruit bien ce qui ressemble à ses exemples d'entraînement, et mal le reste. La cellule suivante reconstruit aussi les images d'entraînement, dans `X_train_pred` ; celles de test sont dans `X_test_pred`. Puis :

- calculez l'erreur quadratique moyenne de reconstruction de chaque image, dans `train_errors` et `test_errors` ;
- fixez le seuil au quantile à 99 % de `train_errors`, et comptez les images de test qui le dépassent ;
- affichez les 10 images de test les plus mal reconstruites au-dessus de leur reconstruction, avec `show_images`, puis les 10 mieux reconstruites.

Les pires sont-elles des chiffres mal écrits, ou des chiffres d'une classe précise ? Et les meilleures ?

**Documentation utile**

- [`torch.quantile`](https://docs.pytorch.org/docs/stable/generated/torch.quantile.html)
- [`torch.argsort`](https://docs.pytorch.org/docs/stable/generated/torch.argsort.html)

In [ ]:
X_train_pred = predict(autoencoder, X_train)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `((X_train - X_train_pred) ** 2).mean(dim=1)` : une erreur par image (la moyenne sur ses 784 pixels)
- `torch.quantile(train_errors, 0.99)` : la valeur que dépassent 1 % des erreurs d'entraînement ; `(test_errors > threshold).sum()` compte les images de test au-delà, de l'ordre de 1 % d'entre elles si le test ressemble à l'entraînement
- `test_errors.argsort(descending=True)` : les indices des images, de la plus grande erreur à la plus petite
- `show_images(X_test[worst], X_test_pred[worst], titles=y_test[worst].tolist())` : les originaux au-dessus de leur reconstruction

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
train_errors = ...
test_errors = ...
threshold = ...
print(f"Seuil : {threshold:.4f}, {...} images de test au-delà")

order = test_errors.argsort(descending=True)
worst, best = order[:10], ...
show_images(X_test[worst], X_test_pred[worst], titles=y_test[worst].tolist())
show_images(...)
```

### Solution

In [ ]:
train_errors = ((X_train - X_train_pred) ** 2).mean(dim=1)
test_errors = ((X_test - X_test_pred) ** 2).mean(dim=1)
threshold = torch.quantile(train_errors, 0.99)
print(
  f"Seuil : {threshold:.4f}, {(test_errors > threshold).sum()} images de test au-delà"
)

order = test_errors.argsort(descending=True)
worst, best = order[:10], order[-10:]
print("Les pires reconstructions")
show_images(X_test[worst], X_test_pred[worst], titles=y_test[worst].tolist())
print("Les meilleures")
show_images(X_test[best], X_test_pred[best], titles=y_test[best].tolist())

Un peu moins de 1 % des images de test dépassent le seuil (environ 75), comme attendu quand le test ressemble à l'entraînement. Les pires reconstructions sont des chiffres de forme rare, surtout des 8 et des 6 aux boucles inhabituelles : ils s'éloignent de ce que l'auto-encodeur a vu, et sa reconstruction les ramène vers un chiffre plus courant. Les meilleures sont toutes des 1 : peu de pixels allumés, une forme simple, donc peu d'erreur. L'erreur dépend donc aussi de la classe : avec un seul seuil, un chiffre compliqué est plus souvent signalé qu'un 1.

## 6. La taille du code

Un code plus grand reconstruit mieux, mais compresse moins. Mesurez ce compromis :

- écrivez `build_autoencoder(encoding_dim)`, qui construit l'auto-encodeur profond de la partie 3 (l'`nn.Sequential` nommé, sans l'`AutoEncoder`) pour une taille de code donnée ;
- entraînez-en un pour chaque taille de code de 2, 4, 8, 16, 32 et 64, pendant 10 epochs chacun, avec la perte de la partie 3 et `train(..., verbose=False)` ;
- gardez chaque réseau entraîné dans un dictionnaire `autoencoders`, et sa dernière perte de validation dans `val_losses`, indexés par la taille du code ;
- tracez la perte de validation en fonction de la taille du code, avec un axe des abscisses en échelle logarithmique de base 2.

Quelle taille de code choisiriez-vous ? Quel taux de compression (784 / taille du code) obtenez-vous alors ?

**Documentation utile**

- [`matplotlib.pyplot.xscale`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.xscale.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- le corps de `build_autoencoder` est la solution de la partie 3, avec `encoding_dim` en paramètre, suivie de `return autoencoder`
- `train(AutoEncoder(autoencoders[size], loss=nn.BCELoss()), max_epochs=10, name=f"size_{size}", verbose=False)` rend la dernière perte de validation ; `autoencoders[size]`, le réseau enveloppé, est entraîné sur place
- la boucle prend quelques minutes : `print` la perte de chaque taille au fil de l'eau
- `plt.plot(list(val_losses), list(val_losses.values()), marker="o")`, puis `plt.xscale("log", base=2)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def build_autoencoder(encoding_dim: int) -> nn.Sequential:
  encoder = ...
  decoder = ...
  return nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))


autoencoders, val_losses = {}, {}
for size in [2, 4, 8, 16, 32, 64]:
  autoencoders[size] = build_autoencoder(size)
  val_losses[size] = train(...)
  print(f"Code de taille {size} : {val_losses[size]:.4f}")

plt.plot(...)
plt.xscale(...)
plt.xlabel("Taille du code")
plt.ylabel("Perte de validation")
plt.show()
```

### Solution

In [ ]:
def build_autoencoder(encoding_dim: int) -> nn.Sequential:
  encoder = nn.Sequential(
    nn.Linear(input_dim, 128),
    nn.ReLU(),
    nn.Linear(128, encoding_dim),
    nn.ReLU(),
  )
  decoder = nn.Sequential(
    nn.Linear(encoding_dim, 128),
    nn.ReLU(),
    nn.Linear(128, input_dim),
    nn.Sigmoid(),
  )
  return nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))


autoencoders, val_losses = {}, {}
for size in [2, 4, 8, 16, 32, 64]:
  autoencoders[size] = build_autoencoder(size)
  val_losses[size] = train(
    AutoEncoder(autoencoders[size], loss=nn.BCELoss()),
    max_epochs=10,
    name=f"size_{size}",
    verbose=False,
  )
  print(
    f"Code de taille {size} (compression {input_dim / size:.0f}x) : "
    f"perte de validation {val_losses[size]:.4f}"
  )

plt.plot(list(val_losses), list(val_losses.values()), marker="o")
plt.xscale("log", base=2)
plt.xlabel("Taille du code")
plt.ylabel("Perte de validation")
plt.show()

La perte baisse vite tant que le code est très petit, puis de moins en moins : de 2 à 4 dimensions, elle perd environ 0,04, de 32 à 64, trois fois moins. Au-delà d'une trentaine de dimensions, agrandir le code n'améliore presque plus la reconstruction. Un code de 16 à 32 dimensions est un bon compromis, soit un taux de compression de 784 / 32 = 24,5 à 784 / 16 = 49 (en nombre de valeurs : le code et les pixels sont tous des flottants).

## 7. Pour aller plus loin : un auto-encodeur épars

Un auto-encodeur épars a un code large, mais une pénalité sur ses activations n'en laisse que peu actives pour chaque image : chaque unité du code finit par porter un motif qu'on peut regarder seul. Construisez-en un :

- `SparseAutoEncoder`, une sous-classe d'`AutoEncoder` dont `_step` ajoute à la perte de reconstruction une pénalité L1 sur les activations du code : `l1` fois la somme de leurs valeurs absolues, en moyenne sur les images du lot, avec `l1 = 1e-3` ;
- `sparse_autoencoder`, l'auto-encodeur profond de la partie 6 à code de 64 unités (`build_autoencoder(64)`), enveloppé dans un `SparseAutoEncoder` avec la perte de la partie 3, et entraîné 10 epochs.

Comparez-le à l'auto-encodeur à code de 64 unités de la partie 6, `autoencoders[64]` : le nombre moyen d'unités actives (> 0) du code par image de test, le nombre d'unités jamais actives, et l'erreur quadratique moyenne de reconstruction. Que coûte la parcimonie ?

**Documentation utile**

- [`torch.Tensor.abs`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.abs.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- dans `_step`, `codes = self.model.encoder(images)`, puis `self.model.decoder(codes)` : la reconstruction, en passant par le code
- la pénalité : `self.l1 * codes.abs().sum(dim=1).mean()`, une somme sur les 64 unités de chaque image, puis une moyenne sur le lot ; c'est ce que fait Keras avec `activity_regularizer=keras.regularizers.L1(1e-3)`
- `__init__` reçoit `l1` en plus, le garde dans `self.l1` et passe le reste à `super().__init__(model, loss)`
- `codes = predict(network.encoder, X_test)`, puis `(codes > 0).sum(dim=1).float().mean()` : le nombre moyen d'unités actives par image ; `(codes.max(dim=0).values == 0).sum()` : les unités jamais actives

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class SparseAutoEncoder(AutoEncoder):
  def __init__(self, model: nn.Module, loss: nn.Module, l1: float) -> None:
    super().__init__(model, loss)
    self.l1 = l1

  def _step(self, batch: tuple[torch.Tensor], stage: str) -> torch.Tensor:
    (images,) = batch
    codes = ...
    loss = self.loss(..., images) + ...
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    return loss


sparse_autoencoder = build_autoencoder(64)
train(SparseAutoEncoder(...), max_epochs=..., name="sparse")

for name, network in [("Sans pénalité", autoencoders[64]), ("L1", sparse_autoencoder)]:
  codes = predict(network.encoder, X_test)
  error = ((X_test - predict(network, X_test)) ** 2).mean()
  print(f"{name} : ...")
```

### Solution

In [ ]:
class SparseAutoEncoder(AutoEncoder):
  """An AutoEncoder whose loss adds an L1 penalty on the code's activations."""

  def __init__(self, model: nn.Module, loss: nn.Module, l1: float) -> None:
    super().__init__(model, loss)
    self.l1 = l1

  def _step(self, batch: tuple[torch.Tensor], stage: str) -> torch.Tensor:
    (images,) = batch
    codes = self.model.encoder(images)
    penalty = self.l1 * codes.abs().sum(dim=1).mean()
    loss = self.loss(self.model.decoder(codes), images) + penalty
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    return loss


sparse_autoencoder = build_autoencoder(64)
train(
  SparseAutoEncoder(sparse_autoencoder, loss=nn.BCELoss(), l1=1e-3),
  max_epochs=10,
  name="sparse",
  verbose=False,
)

for name, network in [("Sans pénalité", autoencoders[64]), ("L1", sparse_autoencoder)]:
  codes = predict(network.encoder, X_test)
  error = ((X_test - predict(network, X_test)) ** 2).mean()
  print(
    f"{name} : {(codes > 0).sum(dim=1).float().mean():.1f} unités actives par image sur 64, "
    f"{(codes.max(dim=0).values == 0).sum()} jamais actives, erreur {error:.4f}"
  )

Avec la pénalité, chaque image n'active plus qu'une vingtaine d'unités du code au lieu d'une soixantaine, et près de la moitié des unités ne s'activent jamais : la pénalité éteint les unités les moins utiles. La parcimonie se paie en reconstruction, dont l'erreur double presque (de 0,007 à 0,012). Le coefficient règle ce compromis : essayez 1e-4 et 1e-2. Dans un code épars, chaque unité active porte un motif qu'on peut regarder seul : c'est ce qui sert à interpréter les représentations des grands modèles de langage.